In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
bundle = next(
    (candidate for path in (cwd, *cwd.parents)
     for candidate in (path, path / "final_code")
     if (candidate / "inference" / "run_models.ipynb").is_file()),
    None,
)
if bundle is None:
    raise FileNotFoundError("Open this notebook from within the project directory.")

notebook_dir = bundle / "inference"
os.chdir(notebook_dir)
sys.path.insert(0, str(notebook_dir))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os
import numpy as np

In [ ]:
def prediction_visualization(df, columns, title,
                        label_col=None, pred_col=None,
                        color_map=None, save=False, outdir="figure"):
    ts = pd.to_datetime(df['timestamp'], errors='coerce')
    valid = ts.notna()
    if not valid.all():
        df = df.loc[valid].reset_index(drop=True)
        ts = ts[valid].reset_index(drop=True)
    if ts.dt.tz is not None:
        ts = ts.dt.tz_convert(None)
    
    feats = [c for c in columns if c != 'timestamp']
    colors = color_map or plt.cm.tab10.colors
    n_feats = len(feats)

    if save:
        os.makedirs(outdir, exist_ok=True)

    # feature 개수 × 2 (actual, pred) 서브플롯 생성
    fig, axes = plt.subplots(n_feats, 2, figsize=(20, 3 * n_feats), sharex=True)
    
    # feature가 1개일 경우 axes 차원 맞추기
    if n_feats == 1:
        axes = axes.reshape(1, -1)

    # --- 연속 1 구간 추출 함수 ---
    def spans_from_binary(s):
        v = s.fillna(0).astype(int).values
        idx = np.flatnonzero(np.diff(np.r_[0, v, 0]) != 0)
        return [(idx[i], idx[i+1]-1) for i in range(0, len(idx), 2)]

    for i, f in enumerate(feats):
        ax_actual = axes[i, 0]
        ax_pred = axes[i, 1]
        color = colors[i % len(colors)]

        # --- 실제값 + label ---
        ax_actual.plot(ts, df[f], lw=0.8, label=f, color=color)
        if label_col and label_col in df.columns:
            for s, e in spans_from_binary(df[label_col]):
                ax_actual.axvspan(ts.iloc[s], ts.iloc[e], color='grey', alpha=0.5, ec='green', lw=1)
        ax_actual.set_title(f"{f} - Actual")
        ax_actual.grid(True, alpha=0.3)
        ax_actual.legend(loc='upper right')

        # --- 예측값 + pred label ---
        p = f"{f}_pred"
        if p in df.columns:
            ax_pred.plot(ts, df[p], lw=0.8, label=p, color=color)
        if pred_col and pred_col in df.columns:
            for s, e in spans_from_binary(df[pred_col]):
                ax_pred.axvspan(ts.iloc[s], ts.iloc[e], color='red', alpha=0.5, ec='red', lw=1)
        ax_pred.set_title(f"{f} - Prediction")
        ax_pred.grid(True, alpha=0.3)
        ax_pred.legend(loc='upper right')

        # --- 두 그래프의 y축 스케일 통일 ---
        ymin = min(ax_actual.get_ylim()[0], ax_pred.get_ylim()[0])
        ymax = max(ax_actual.get_ylim()[1], ax_pred.get_ylim()[1])
        ax_actual.set_ylim(ymin, ymax)
        ax_pred.set_ylim(ymin, ymax)

    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    if save:
        plt.savefig(os.path.join(outdir, f"{title}.png"), dpi=150)
        plt.close(fig)
    else:
        plt.show()

#### Chronos-t5-large(✔)


In [ ]:
from run_chronos_t5_large import run_chronos_t5_large

##### Lab Data

In [ ]:
from run_chronos_t5_large import run_chronos_t5_large

# 3st_collection
usecols=[
    'timestamp',
    'accel_x', 'accel_y', 'accel_z',
    'gyro_x', 'gyro_y', 'gyro_z',
    'mag_x', 'mag_y', 'mag_z',
    'label',
    ]

model = 'chronos_t5_large'

###### **Zeroshot**

In [ ]:
for abnormal in ['instop']:#, 'grabbing', 'hitting',]:
    # Load data
    df = pd.read_csv(f'../data/3st_collection/labeled_abnor_imu_{abnormal}.csv', usecols=usecols)
    result_df = run_chronos_t5_large(df=df, abnormal=abnormal)
    # Save result
    save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"{model}_3st_output.csv")
    result_df.to_csv(save_path, index=False)
    print(f"Saved to {save_path}")

###### **Fine-tuning**

In [ ]:
for abnormal in ['instop']:#, 'grabbing', 'hitting',]:
    df = pd.read_csv(f'../data/3st_collection/labeled_abnor_imu_{abnormal}.csv', usecols=usecols)
    ckpt_path="../finetuning/weights/chronos-t5-large/chronos-t5-large_full_finetuned"
    result_df = run_chronos_t5_large(df=df, abnormal=abnormal, ckpt_path=ckpt_path)

    save_dir = f"../results/inference_df/full_finetuned/{abnormal}_all_scenario/"
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"{model}_3st_output.csv")
    result_df.to_csv(save_path, index=False)
    print(f"Saved to {save_path}")

##### DS Data

###### **Zeroshot**

In [ ]:
import pandas as pd
import os

model = 'chronos_t5_large'

# Ds_collection
usecols=[
    'timestamp',
    'acc_x', 'acc_y', 'acc_z',
    'gyro_x', 'gyro_y', 'gyro_z'
    ]

ds_press_imu = pd.read_csv('../data/ds_collection/ds_press_imu.csv')
date = "2025-11-17"
target_date = pd.to_datetime(date).date()
df = ds_press_imu[ds_press_imu["timestamp"].str.contains(date)].sort_values("timestamp")
df = df[usecols]

abnormal = 'ds_press_2025-11-17'
result_df = run_chronos_t5_large(df=df, abnormal=abnormal)
# Save result
save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_ds_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

###### **Finetuned**

In [ ]:
import pandas as pd
import os
model = 'chronos-t5-large'

# Ds_collection
usecols=[
    'timestamp',
    'acc_x', 'acc_y', 'acc_z',
    'gyro_x', 'gyro_y', 'gyro_z'
    ]

ds_press_imu = pd.read_csv('../data/ds_collection/ds_press_imu.csv')
date = "2025-11-17"
target_date = pd.to_datetime(date).date()
df = ds_press_imu[ds_press_imu["timestamp"].str.contains(date)].sort_values("timestamp")
df = df[usecols]

abnormal = 'ds_press_2025-11-17'
ckpt_path="../finetuning/weights/chronos-t5-large/chronos-t5-large_ds_full_finetuned"
result_df = run_chronos_t5_large(df=df, abnormal=abnormal, ckpt_path=ckpt_path)
# Save result
save_dir = f"../results/inference_df/full_finetuned/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_ds_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

###### **Check**

In [ ]:
abnormal = 'ds_press_2025-11-17'

df = pd.read_csv(f"../results/inference_df/zeroshot/{abnormal}_all_scenario/chronos_t5_large_ds_output.csv")
finetuned_df = pd.read_csv(f"../results/inference_df/full_finetuned/ds_press_2025-11-17_all_scenario/chronos_t5_large_ds_output.csv")

df = df[50000:55000]
finetuned_df = finetuned_df[50000:55000]

prediction_visualization(
    df,
    columns=['timestamp','acc_x','acc_y','acc_z','gyro_x','gyro_y','gyro_z',],#'acc_y','acc_z','gyro_y'
    title='Chronos_t5_large DS',
    # label_col='label',
    )
prediction_visualization(
    finetuned_df,
    columns=['timestamp','acc_x','acc_y','acc_z','gyro_x','gyro_y','gyro_z',],#'acc_y','acc_z','gyro_y'
    title='Chronos_t5_large DS (Finetuned)',
    # label_col='label',
    )

##### CASPER

###### **Zeroshot**

In [ ]:
import pandas as pd
import os
model = 'chronos_t5_large'

df = pd.read_csv('../data/casper.csv')
df.rename(columns={'lable': 'label'}, inplace=True)

# casper_collection
usecols=[
    'accel_x', 'accel_y', 'accel_z',
    'gyro_x', 'gyro_y', 'gyro_z',
    'mag_x', 'mag_y', 'mag_z',
    'label',
    ]

df = df[usecols]

abnormal = 'casper'
result_df = run_chronos_t5_large(df=df, abnormal=abnormal)
# Save result
save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_benchmark_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

##### CASPER 2

###### **Normal Data**

In [ ]:
import pandas as pd
import os
model = 'chronos_t5_large'

df = pd.read_csv('../data/IMU_10Hz_task_labeled.csv')
df_label = df

df.rename(columns={
    "accX": "accel_x","accY": "accel_y","accZ": "accel_z",
    "gyroX": "gyro_x","gyroY": "gyro_y","gyroZ": "gyro_z",
    "magX": "mag_x","magY": "mag_y","magZ": "mag_z"
    }, inplace=True)

# casper_collection
usecols=[
    'accel_x', 'accel_y', 'accel_z',
    'gyro_x', 'gyro_y', 'gyro_z',
    'mag_x', 'mag_y', 'mag_z',
    ]

df = df[usecols]
df = df[:100000]

abnormal = 'casper2'
result_df = run_chronos_t5_large(df=df, abnormal=abnormal)

cd_label = df_label[512:]['cd_label']
result_df['cd_label'] = cd_label

# Save result
save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_normal_benchmark_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

###### **hitting**

In [ ]:
import pandas as pd
import os
model = 'chronos_t5_large'

df = pd.read_csv('../data/IMU_hitting_arm_task_labeled.csv')

df.rename(columns={
    "accX": "accel_x","accY": "accel_y","accZ": "accel_z",
    "gyroX": "gyro_x","gyroY": "gyro_y","gyroZ": "gyro_z",
    "magX": "mag_x","magY": "mag_y","magZ": "mag_z"
    }, inplace=True)

# casper_collection
usecols=[
    'accel_x', 'accel_y', 'accel_z',
    'gyro_x', 'gyro_y', 'gyro_z',
    'mag_x', 'mag_y', 'mag_z',
    ]

df = df[usecols]

abnormal = 'casper2'
result_df = run_chronos_t5_large(df=df, abnormal=abnormal)
# Save result
save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_benchmark_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

##### RoAD

In [ ]:
import pandas as pd
import os

usecols = [
    'sensor_id5_accel_x','sensor_id5_accel_y','sensor_id5_accel_z',
    'sensor_id5_gyro_x','sensor_id5_gyro_y','sensor_id5_gyro_z',
    'sensor_id6_accel_x','sensor_id6_accel_y','sensor_id6_accel_z',
    'sensor_id6_gyro_x','sensor_id6_gyro_y','sensor_id6_gyro_z',
    'label'
    ]
model = 'chronos_t5_large'
df = pd.read_csv("../data/road_collision.csv", usecols=usecols)

abnormal = 'road_collision'
result_df = run_chronos_t5_large(df=df, abnormal=abnormal)
# Save result
save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_benchmark_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

#### Chronos-2(✔)

In [ ]:
from run_chronos2 import run_chronos2

##### Lab Data

In [ ]:
# 3st_collection
usecols=[
    'timestamp',
    'accel_x', 'accel_y', 'accel_z',
    'gyro_x', 'gyro_y', 'gyro_z',
    'mag_x', 'mag_y', 'mag_z',
    'label',
    ]

model = 'chronos2'

###### **Zeroshot**

In [ ]:
for abnormal in ['instop']:#, 'grabbing', 'hitting',]:
    df = pd.read_csv(f'../data/3st_collection/labeled_abnor_imu_{abnormal}.csv', usecols=usecols)
    result_df = run_chronos2(df=df, abnormal=abnormal)
    
    save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"{model}_3st_output.csv")
    result_df.to_csv(save_path, index=False)
    print(f"Saved to {save_path}")

###### **Fine-tuning**

In [ ]:
for abnormal in ['instop']:#, 'grabbing', 'hitting',]:
    df = pd.read_csv(f'../data/3st_collection/labeled_abnor_imu_{abnormal}.csv', usecols=usecols)
    ckpt_path="../finetuning/weights/chronos2/chronos2_full_finetuned"
    result_df = run_chronos2(df=df, abnormal=abnormal, ckpt_path=ckpt_path)

    save_dir = f"../results/inference_df/full_finetuned/{abnormal}_all_scenario/"
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"{model}_3st_output.csv")
    result_df.to_csv(save_path, index=False)
    print(f"Saved to {save_path}")

In [ ]:
df = pd.read_csv("../results/inference_df/zeroshot/instop_all_scenario/chronos2_3st_output.csv")
finetuned_df = pd.read_csv("../results/inference_df/full_finetuned/instop_all_scenario/chronos2_3st_output.csv")

prediction_visualization(
    df,
    columns=['timestamp', 'accel_x','accel_y','accel_z'],
    title='Chronos2 Instop',
    label_col='label',
    )
prediction_visualization(
    finetuned_df,
    columns=['timestamp', 'accel_x','accel_y','accel_z'],
    title='Chronos2 Instop (Finetuned)',
    label_col='label',
    )

##### DS Data

###### **Zeroshot**

In [ ]:
import pandas as pd
import os
model = 'chronos2'

# Ds_collection
usecols=[
    'timestamp',
    'acc_x', 'acc_y', 'acc_z',
    'gyro_x', 'gyro_y', 'gyro_z'
    ]

ds_press_imu = pd.read_csv('../data/ds_collection/ds_press_imu.csv')
date = "2025-11-15"
target_date = pd.to_datetime(date).date()
df = ds_press_imu[ds_press_imu["timestamp"].str.contains(date)].sort_values("timestamp")
df = df[usecols]

In [ ]:
abnormal = 'ds_press_2025-11-15'
result_df = run_chronos2(df=df, abnormal=abnormal)
# Save result
save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_ds_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

###### **Finetuned**

In [ ]:
import pandas as pd
import os
model = 'chronos2'

# Ds_collection
usecols=[
    'timestamp',
    'acc_x', 'acc_y', 'acc_z',
    'gyro_x', 'gyro_y', 'gyro_z'
    ]

ds_press_imu = pd.read_csv('../data/ds_collection/ds_press_imu.csv')
date = "2025-11-17"
target_date = pd.to_datetime(date).date()
df = ds_press_imu[ds_press_imu["timestamp"].str.contains(date)].sort_values("timestamp")
df = df[usecols]

In [ ]:
abnormal = 'ds_press_2025-11-17'
ckpt_path="../finetuning/weights/chronos2/chronos2_ds_lora_finetuned"
result_df = run_chronos2(df=df, abnormal=abnormal, ckpt_path=ckpt_path)
# Save result
save_dir = f"../results/inference_df/lora_finetuned/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_ds_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

###### **Check**

In [ ]:
abnormal = 'ds_press_2025-11-17'

df = pd.read_csv(f"../results/inference_df/zeroshot/{abnormal}_all_scenario/chronos2_ds_output.csv")
finetuned_df = pd.read_csv(f"../results/inference_df/lora_finetuned/{abnormal}_all_scenario/chronos2_ds_output.csv")

df = df[50000:55000]
finetuned_df = finetuned_df[50000:55000]

prediction_visualization(
    df,
    columns=['timestamp','acc_x','acc_y','acc_z','gyro_x','gyro_y','gyro_z',],#'acc_y','acc_z','gyro_y'
    title='Chronos2 DS',
    # label_col='label',
    )
prediction_visualization(
    finetuned_df,
    columns=['timestamp','acc_x','acc_y','acc_z','gyro_x','gyro_y','gyro_z',],#'acc_y','acc_z','gyro_y'
    title='Chronos2 DS (Finetuned)',
    # label_col='label',
    )

##### CASPER

###### **Zeroshot**

In [ ]:
import pandas as pd
import os
model = 'chronos2'

df = pd.read_csv('../data/casper.csv')
df.rename(columns={'lable': 'label'}, inplace=True)

# casper_collection
usecols=[
    'accel_x', 'accel_y', 'accel_z',
    'gyro_x', 'gyro_y', 'gyro_z',
    'mag_x', 'mag_y', 'mag_z',
    'label',
    ]

df = df[usecols]

abnormal = 'casper'
result_df = run_chronos2(df=df, abnormal=abnormal)
# Save result
save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_benchmark_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

In [ ]:
abnormal = 'casper'

df = pd.read_csv(f"../results/inference_df/zeroshot/{abnormal}_all_scenario/chronos2_benchmark_output.csv")
# print(len(df))

# df = df[:100000]
prediction_visualization(
    df,
    columns=['accel_x','accel_y','accel_z','gyro_x','gyro_y','gyro_z','accel_y','accel_z','gyro_y'],
    title='Chronos2 CASPER',
    label_col='label',
    )

In [ ]:
df = df[:100000]
prediction_visualization(
    df,
    columns=['accel_x','accel_y','accel_z','gyro_x','gyro_y','gyro_z','accel_y','accel_z','gyro_y'],
    title='Chronos2 CASPER',
    label_col='label',
    )

In [ ]:

# finetuned_df = pd.read_csv(f"../results/inference_df/lora_finetuned/{abnormal}_all_scenario/chronos2_ds_output.csv")

# df = df[50000:55000]
# finetuned_df = finetuned_df[50000:55000]

prediction_visualization(
    df,
    columns=['accel_x','accel_y','accel_z','gyro_x','gyro_y','gyro_z','accel_y','accel_z','gyro_y'],
    title='Chronos2 CASPER',
    label_col='label',
    )

##### CASPER2

###### **Normal Data**

In [ ]:
import pandas as pd
import os
model = 'chronos2'

df = pd.read_csv('../data/IMU_10Hz_task_labeled.csv')
df_label = df

df.rename(columns={
    "accX": "accel_x","accY": "accel_y","accZ": "accel_z",
    "gyroX": "gyro_x","gyroY": "gyro_y","gyroZ": "gyro_z",
    "magX": "mag_x","magY": "mag_y","magZ": "mag_z"
    }, inplace=True)

# casper_collection
usecols=[
    'accel_x', 'accel_y', 'accel_z',
    'gyro_x', 'gyro_y', 'gyro_z',
    'mag_x', 'mag_y', 'mag_z',
    ]

df = df[usecols]
df = df[:100000]

abnormal = 'casper2'
result_df = run_chronos2(df=df, abnormal=abnormal)

cd_label = df_label[512:]['cd_label']
result_df['cd_label'] = cd_label

# Save result
save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_normal_benchmark_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

###### **hitting**

In [ ]:
import pandas as pd
import os
model = 'chronos2'

df = pd.read_csv('../data/IMU_hitting_arm_task_labeled.csv')

df.rename(columns={
    "accX": "accel_x","accY": "accel_y","accZ": "accel_z",
    "gyroX": "gyro_x","gyroY": "gyro_y","gyroZ": "gyro_z",
    "magX": "mag_x","magY": "mag_y","magZ": "mag_z"
    }, inplace=True)

# casper_collection
usecols=[
    'accel_x', 'accel_y', 'accel_z',
    'gyro_x', 'gyro_y', 'gyro_z',
    'mag_x', 'mag_y', 'mag_z',
    ]

df = df[usecols]

abnormal = 'casper2'
result_df = run_chronos2(df=df, abnormal=abnormal)
# Save result
save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_benchmark_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

##### RoAD

In [ ]:
import pandas as pd
import os

usecols = [
    'sensor_id5_accel_x','sensor_id5_accel_y','sensor_id5_accel_z',
    'sensor_id5_gyro_x','sensor_id5_gyro_y','sensor_id5_gyro_z',
    'sensor_id6_accel_x','sensor_id6_accel_y','sensor_id6_accel_z',
    'sensor_id6_gyro_x','sensor_id6_gyro_y','sensor_id6_gyro_z',
    'label'
    ]
model = 'chronos2'
df = pd.read_csv("../data/road_collision.csv", usecols=usecols)

abnormal = 'road_collision'
result_df = run_chronos2(df=df, abnormal=abnormal)
# Save result
save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_benchmark_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

#### TimesFM(✔)

In [ ]:
from run_timesfm import run_timesfm

##### Lab Data

In [ ]:
# 3st_collection
usecols=[
    'timestamp',
    'accel_x', 'accel_y', 'accel_z',
    'gyro_x', 'gyro_y', 'gyro_z',
    'mag_x', 'mag_y', 'mag_z',
    'label'
    ]

model = 'timesfm'

###### **Zeroshot**

In [ ]:
for abnormal in ['instop', 'grabbing', 'hitting',]:
    df = pd.read_csv(f'../data/3st_collection/labeled_abnor_imu_{abnormal}.csv', usecols=usecols)
    result_df = run_timesfm(df=df, abnormal=abnormal)
    
    save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"{model}_3st_output.csv")
    result_df.to_csv(save_path, index=False)
    print(f"Saved to {save_path}")

###### **Fine-tuning**

In [ ]:
for abnormal in ['instop', 'grabbing', 'hitting',]:
    df = pd.read_csv(f'../data/3st_collection/labeled_abnor_imu_{abnormal}.csv', usecols=usecols)
    ckpt_path="../finetuning/weights/timesfm/timesfm_full_finetuned/timesfm_full_finetuned.safetensors"
    result_df = run_timesfm(df=df, abnormal=abnormal, ckpt_path=ckpt_path)

    save_dir = f"../results/inference_df/full_finetuned/{abnormal}_all_scenario/"
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"{model}_3st_output.csv")
    result_df.to_csv(save_path, index=False)
    print(f"Saved to {save_path}")

##### DS Data

###### **Zeroshot**

In [ ]:
import pandas as pd
import os
model = 'timesfm'

# Ds_collection
usecols=[
    'timestamp',
    'acc_x', 'acc_y', 'acc_z',
    'gyro_x', 'gyro_y', 'gyro_z'
    ]

ds_press_imu = pd.read_csv('../data/ds_collection/ds_press_imu.csv')
date = "2025-11-17"
target_date = pd.to_datetime(date).date()
df = ds_press_imu[ds_press_imu["timestamp"].str.contains(date)].sort_values("timestamp")
df = df[usecols]

abnormal = 'ds_press_2025-11-17'
result_df = run_timesfm(df=df, abnormal=abnormal)
# Save result
save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_ds_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

###### **Finetuning**

In [ ]:
import pandas as pd
import os
model = 'timesfm'

# Ds_collection
usecols=[
    'timestamp',
    'acc_x', 'acc_y', 'acc_z',
    'gyro_x', 'gyro_y', 'gyro_z'
    ]

ds_press_imu = pd.read_csv('../data/ds_collection/ds_press_imu.csv')
date = "2025-11-17"
target_date = pd.to_datetime(date).date()
df = ds_press_imu[ds_press_imu["timestamp"].str.contains(date)].sort_values("timestamp")
df = df[usecols]

abnormal = 'ds_press_2025-11-17'
ckpt_path="../finetuning/weights/timesfm/timesfm_ds_full_finetuned/timesfm_full_finetuned.safetensors"
result_df = run_timesfm(df=df, abnormal=abnormal, ckpt_path=ckpt_path)
# Save result
save_dir = f"../results/inference_df/full_finetuned/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_ds_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

###### **Check**

In [ ]:
abnormal = 'ds_press_2025-11-17'

df = pd.read_csv(f"../results/inference_df/zeroshot/{abnormal}_all_scenario/timesfm_ds_output.csv")
finetuned_df = pd.read_csv(f"../results/inference_df/full_finetuned/ds_press_2025-11-17_all_scenario/timesfm_ds_output.csv")

df = df[50000:55000]
finetuned_df = finetuned_df[50000:55000]

prediction_visualization(
    df,
    columns=['timestamp','acc_x','acc_y','acc_z','gyro_x','gyro_y','gyro_z',],#'acc_y','acc_z','gyro_y'
    title='TimesFM DS',
    # label_col='label',
    )
prediction_visualization(
    finetuned_df,
    columns=['timestamp','acc_x','acc_y','acc_z','gyro_x','gyro_y','gyro_z',],#'acc_y','acc_z','gyro_y'
    title='TimesFM DS (Finetuned)',
    # label_col='label',
    )

##### CASPER

###### **Zeroshot**

In [ ]:
import pandas as pd
import os
model = 'timesfm'

df = pd.read_csv('../data/casper.csv')
df.rename(columns={'lable': 'label'}, inplace=True)

# casper_collection
usecols=[
    'accel_x', 'accel_y', 'accel_z',
    'gyro_x', 'gyro_y', 'gyro_z',
    'mag_x', 'mag_y', 'mag_z',
    'label',
    ]

df = df[usecols]

abnormal = 'casper'
result_df = run_timesfm(df=df, abnormal=abnormal)
# Save result
save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_benchmark_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

##### CASPER2

###### **Normal Data**

In [ ]:
import pandas as pd
import os
model = 'timesfm'

df = pd.read_csv('../data/IMU_10Hz_task_labeled.csv')
df_label = df

df.rename(columns={
    "accX": "accel_x","accY": "accel_y","accZ": "accel_z",
    "gyroX": "gyro_x","gyroY": "gyro_y","gyroZ": "gyro_z",
    "magX": "mag_x","magY": "mag_y","magZ": "mag_z"
    }, inplace=True)

# casper_collection
usecols=[
    'accel_x', 'accel_y', 'accel_z',
    'gyro_x', 'gyro_y', 'gyro_z',
    'mag_x', 'mag_y', 'mag_z',
    ]

df = df[usecols]
df = df[:100000]

abnormal = 'casper2'
result_df = run_timesfm(df=df, abnormal=abnormal)

cd_label = df_label[512:]['cd_label']
result_df['cd_label'] = cd_label

# Save result
save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_normal_benchmark_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

###### **hitting**

In [ ]:
import pandas as pd
import os
model = 'timesfm'

df = pd.read_csv('../data/IMU_hitting_arm_task_labeled.csv')

df.rename(columns={
    "accX": "accel_x","accY": "accel_y","accZ": "accel_z",
    "gyroX": "gyro_x","gyroY": "gyro_y","gyroZ": "gyro_z", 
    "magX": "mag_x","magY": "mag_y","magZ": "mag_z"
    }, inplace=True)

# casper_collection
usecols=[
    'accel_x', 'accel_y', 'accel_z',
    'gyro_x', 'gyro_y', 'gyro_z',
    'mag_x', 'mag_y', 'mag_z',
    ]

df = df[usecols]

abnormal = 'casper2'
result_df = run_timesfm(df=df, abnormal=abnormal)
# Save result
save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_benchmark_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

##### RoAD

In [ ]:
import pandas as pd
import os

usecols = [
    'sensor_id5_accel_x','sensor_id5_accel_y','sensor_id5_accel_z',
    'sensor_id5_gyro_x','sensor_id5_gyro_y','sensor_id5_gyro_z',
    'sensor_id6_accel_x','sensor_id6_accel_y','sensor_id6_accel_z',
    'sensor_id6_gyro_x','sensor_id6_gyro_y','sensor_id6_gyro_z',
    'label'
    ]
model = 'timesfm'
df = pd.read_csv("../data/road_collision.csv", usecols=usecols)

abnormal = 'road_collision'
result_df = run_timesfm(df=df, abnormal=abnormal)
# Save result
save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_benchmark_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

#### Lagllama(✔)

In [ ]:
from run_lagllama import run_lagllama

##### Lab Data

In [ ]:
# 3st_collection
usecols=[
    'timestamp',
    'accel_x', 'accel_y', 'accel_z',
    'gyro_x', 'gyro_y', 'gyro_z',
    'mag_x', 'mag_y', 'mag_z',
    'label']

model = 'lagllama'

###### **Zeroshot**

In [ ]:
for abnormal in ['instop', 'grabbing', 'hitting',]:
    df = pd.read_csv(f'../data/3st_collection/labeled_abnor_imu_{abnormal}.csv', usecols=usecols)
    result_df = run_lagllama(df=df, abnormal=abnormal)
    
    save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"{model}_3st_output.csv")
    result_df.to_csv(save_path, index=False)
    print(f"Saved to {save_path}")

###### **Fine-tuning**

In [ ]:
for abnormal in ['instop', 'grabbing', 'hitting',]:
    df = pd.read_csv(f'../data/3st_collection/labeled_abnor_imu_{abnormal}.csv', usecols=usecols)
    ckpt_path="../finetuning/weights/lagllama/lagllama_full_finetuned"
    result_df = run_lagllama(df=df, abnormal=abnormal, ckpt_path=ckpt_path)

    save_dir = f"../results/inference_df/full_finetuned/{abnormal}_all_scenario/"
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"{model}_3st_output.csv")
    result_df.to_csv(save_path, index=False)
    print(f"Saved to {save_path}")

##### DS Data

###### **Zeroshot**

In [ ]:
import pandas as pd
import os
model = 'lagllama'

# Ds_collection
usecols=[
    'timestamp',
    'acc_x', 'acc_y', 'acc_z',
    'gyro_x', 'gyro_y', 'gyro_z'
    ]

ds_press_imu = pd.read_csv('../data/ds_collection/ds_press_imu.csv')
date = "2025-11-17"
target_date = pd.to_datetime(date).date()
df = ds_press_imu[ds_press_imu["timestamp"].str.contains(date)].sort_values("timestamp")
df = df[usecols]

abnormal = 'ds_press_2025-11-17'
result_df = run_lagllama(df=df, abnormal=abnormal)
# Save result
save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_ds_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

###### **Finetuned**

In [ ]:
import pandas as pd
import os
model = 'lagllama'

# Ds_collection
usecols=[
    'timestamp',
    'acc_x', 'acc_y', 'acc_z',
    'gyro_x', 'gyro_y', 'gyro_z'
    ]

ds_press_imu = pd.read_csv('../data/ds_collection/ds_press_imu.csv')
date = "2025-11-17"
target_date = pd.to_datetime(date).date()
df = ds_press_imu[ds_press_imu["timestamp"].str.contains(date)].sort_values("timestamp")
df = df[usecols]

abnormal = 'ds_press_2025-11-17'
ckpt_path="../finetuning/weights/lagllama/lagllama_ds_full_finetuned"
result_df = run_lagllama(df=df, abnormal=abnormal, ckpt_path=ckpt_path)
# Save result
save_dir = f"../results/inference_df/full_finetuned/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_ds_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

###### **Check**

In [ ]:
abnormal = 'ds_press_2025-11-17'

df = pd.read_csv(f"../results/inference_df/zeroshot/{abnormal}_all_scenario/lagllama_ds_output.csv")
# finetuned_df = pd.read_csv(f"../results/inference_df/full_finetuned/{abnormal}_all_scenario/lagllama_ds_output.csv")

# df = df[50000:55000]
# finetuned_df = finetuned_df[50000:55000]

prediction_visualization(
    df,
    columns=['timestamp','acc_x','acc_y','acc_z','gyro_x','gyro_y','gyro_z',],#'acc_y','acc_z','gyro_y'
    title='LagLlama DS',
    # label_col='label',
    )
# prediction_visualization(
#     finetuned_df,
#     columns=['timestamp','acc_x','acc_y','acc_z','gyro_x','gyro_y','gyro_z',],#'acc_y','acc_z','gyro_y'
#     title='LagLlama DS (Finetuned)',
#     # label_col='label',
#     )

##### CASPER

###### **Zeroshot**

In [ ]:
import pandas as pd
import os
model = 'lagllama'

df = pd.read_csv('../data/casper.csv')
df.rename(columns={'lable': 'label'}, inplace=True)
df['timestamp'] = pd.date_range(
    start="2024-01-01",
    periods=len(df),
    freq="50ms"   # 20 Hz → 1/20 = 0.05s
)

# casper_collection
usecols=[
    'timestamp',
    'accel_x', 'accel_y', 'accel_z',
    'gyro_x', 'gyro_y', 'gyro_z',
    'mag_x', 'mag_y', 'mag_z',
    'label',
    ]

df = df[usecols]


abnormal = 'casper'
result_df = run_lagllama(df=df, abnormal=abnormal)
# Save result
save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_benchmark_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

##### CASPER2

###### **Normal Data**

In [ ]:
import pandas as pd
import os
model = 'lagllama'

df = pd.read_csv('../data/IMU_10Hz_task_labeled.csv')
df_label = df

df['timestamp'] = pd.date_range(
    start="2024-01-01",
    periods=len(df),
    freq="25ms"
)

df.rename(columns={
    "accX": "accel_x","accY": "accel_y","accZ": "accel_z",
    "gyroX": "gyro_x","gyroY": "gyro_y","gyroZ": "gyro_z",
    "magX": "mag_x","magY": "mag_y","magZ": "mag_z"
    }, inplace=True)

# casper_collection
usecols=[
    'timestamp',
    'accel_x', 'accel_y', 'accel_z',
    'gyro_x', 'gyro_y', 'gyro_z',
    'mag_x', 'mag_y', 'mag_z',
    ]

df = df[usecols]
df = df[:100000]

abnormal = 'casper2'
result_df = run_lagllama(df=df, abnormal=abnormal)

cd_label = df_label[512:]['cd_label']
result_df['cd_label'] = cd_label

# Save result
save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_normal_benchmark_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

In [ ]:
import pandas as pd
df = pd.read_csv('../data/IMU_10Hz_task_labeled.csv')
df_label = df
df_label = df[:100000]

cd_label = df_label[512:]['cd_label']
inference_df = pd.read_csv("../results/inference_df/zeroshot/casper2_all_scenario/lagllama_normal_benchmark_output.csv")

inference_df['cd_label'] = cd_label.values
inference_df.to_csv("../results/inference_df/zeroshot/casper2_all_scenario/lagllama_normal_benchmark_output.csv")

###### **hitting**

In [ ]:
import pandas as pd
import os
model = 'lagllama'

df = pd.read_csv('../data/IMU_hitting_arm_task_labeled.csv')

df['timestamp'] = pd.date_range(
    start="2024-01-01",
    periods=len(df),
    freq="25ms"
)

df.rename(columns={
    "accX": "accel_x","accY": "accel_y","accZ": "accel_z",
    "gyroX": "gyro_x","gyroY": "gyro_y","gyroZ": "gyro_z",
    "magX": "mag_x","magY": "mag_y","magZ": "mag_z"
    }, inplace=True)

# casper_collection
usecols=[
    'timestamp',
    'accel_x', 'accel_y', 'accel_z',
    'gyro_x', 'gyro_y', 'gyro_z',
    'mag_x', 'mag_y', 'mag_z',
    ]

df = df[usecols]

abnormal = 'casper2'
result_df = run_lagllama(df=df, abnormal=abnormal)
# Save result
save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_benchmark_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

##### RoAD

In [ ]:
import pandas as pd
import os

usecols = [
    'sensor_id5_accel_x','sensor_id5_accel_y','sensor_id5_accel_z',
    'sensor_id5_gyro_x','sensor_id5_gyro_y','sensor_id5_gyro_z',
    'sensor_id6_accel_x','sensor_id6_accel_y','sensor_id6_accel_z',
    'sensor_id6_gyro_x','sensor_id6_gyro_y','sensor_id6_gyro_z',
    'label'
    ]
model = 'lagllama'
df = pd.read_csv("../data/road_collision.csv", usecols=usecols)
df['timestamp'] = pd.date_range(
    start="2024-01-01",
    periods=len(df),
    freq="100ms"
)

abnormal = 'road_collision'
result_df = run_lagllama(df=df, abnormal=abnormal)
# Save result
save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_benchmark_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

#### MOMENT(✔)

In [ ]:
from run_moment import run_moment

##### Lab Data

In [ ]:
# 3st_collection
usecols=[
    'timestamp',
    'accel_x', 'accel_y', 'accel_z',
    'gyro_x', 'gyro_y', 'gyro_z',
    'mag_x', 'mag_y', 'mag_z',
    'label']

model = 'moment'

###### **Zeroshot**

In [ ]:
for abnormal in ['instop']:#, 'grabbing', 'hitting',]:
    df = pd.read_csv(f'../data/3st_collection/labeled_abnor_imu_{abnormal}.csv', usecols=usecols)
    result_df = run_moment(df=df, abnormal=abnormal)
    
    save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"{model}_3st_output.csv")
    result_df.to_csv(save_path, index=False)
    print(f"Saved to {save_path}")

###### **Fine-tuning**

In [ ]:
for abnormal in ['instop']:#, 'grabbing', 'hitting',]:
    df = pd.read_csv(f'../data/3st_collection/labeled_abnor_imu_{abnormal}.csv', usecols=usecols)
    ckpt_path="../finetuning/weights/moment/moment_full_finetuned/moment_finetuned.pth"#
    result_df = run_moment(df=df, abnormal=abnormal, ckpt_path=ckpt_path)

    save_dir = f"../results/inference_df/full_finetuned/{abnormal}_all_scenario/"
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"{model}_3st_output.csv")
    result_df.to_csv(save_path, index=False)
    print(f"Saved to {save_path}")

In [ ]:
df = pd.read_csv("../results/inference_df/zeroshot/instop_all_scenario/moment_3st_output.csv")
finetuned_df = pd.read_csv("../results/inference_df/full_finetuned/instop_all_scenario/moment_3st_output.csv")

prediction_visualization(
    df,
    columns=['timestamp', 'accel_x','accel_y','accel_z'],
    title='Chronos2 Instop',
    label_col='label',
    )
prediction_visualization(
    finetuned_df,
    columns=['timestamp', 'accel_x','accel_y','accel_z'],
    title='Chronos2 Instop (Finetuned)',
    label_col='label',
    )

##### DS Data

###### **Zeroshot**

In [ ]:
import pandas as pd
import os
model = 'moment'

# Ds_collection
usecols=[
    'timestamp',
    'acc_x', 'acc_y', 'acc_z',
    'gyro_x', 'gyro_y', 'gyro_z'
    ]

ds_press_imu = pd.read_csv('../data/ds_collection/ds_press_imu.csv')
date = "2025-11-17"
target_date = pd.to_datetime(date).date()
df = ds_press_imu[ds_press_imu["timestamp"].str.contains(date)].sort_values("timestamp")
df = df[usecols]

abnormal = 'ds_press_2025-11-17'
result_df = run_moment(df=df, abnormal=abnormal)
# Save result
save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_ds_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

###### **Finetuned**

In [ ]:
import pandas as pd
import os
model = 'moment'

# Ds_collection
usecols=[
    'timestamp',
    'acc_x', 'acc_y', 'acc_z',
    'gyro_x', 'gyro_y', 'gyro_z'
    ]

ds_press_imu = pd.read_csv('../data/ds_collection/ds_press_imu.csv')
date = "2025-11-17"
target_date = pd.to_datetime(date).date()
df = ds_press_imu[ds_press_imu["timestamp"].str.contains(date)].sort_values("timestamp")
df = df[usecols]

In [ ]:
abnormal = 'ds_press_2025-11-17'
ckpt_path="../finetuning/weights/moment/moment_ds_full_finetuned/moment_finetuned.pth"
result_df = run_moment(df=df, abnormal=abnormal, ckpt_path=ckpt_path)
# Save result
save_dir = f"../results/inference_df/full_finetuned/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_ds_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

###### **Check**

In [ ]:
abnormal = 'ds_press_2025-11-17'

df = pd.read_csv(f"../results/inference_df/zeroshot/{abnormal}_all_scenario/moment_ds_output.csv")
finetuned_df = pd.read_csv(f"../results/inference_df/full_finetuned/{abnormal}_all_scenario/moment_ds_output.csv")

df = df[50000:55000]
finetuned_df = finetuned_df[50000:55000]

prediction_visualization(
    df,
    columns=['timestamp','acc_x','acc_y','acc_z','gyro_x','gyro_y','gyro_z',],#'acc_y','acc_z','gyro_y'
    title='MOMENT DS',
    # label_col='label',
    )
prediction_visualization(
    finetuned_df,
    columns=['timestamp','acc_x','acc_y','acc_z','gyro_x','gyro_y','gyro_z',],#'acc_y','acc_z','gyro_y'
    title='MOMENT DS (Finetuned)',
    # label_col='label',
    )

##### CASPER

###### **Zeroshot**

In [ ]:
import pandas as pd
import os
model = 'moment'

df = pd.read_csv('../data/casper.csv')
df.rename(columns={'lable': 'label'}, inplace=True)

# casper_collection
usecols=[
    'accel_x', 'accel_y', 'accel_z',
    'gyro_x', 'gyro_y', 'gyro_z',
    'mag_x', 'mag_y', 'mag_z',
    'label',
    ]

df = df[usecols]

abnormal = 'casper'
result_df = run_moment(df=df, abnormal=abnormal)
# Save result
save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_benchmark_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

##### CASPER2

###### **Normal Data**

In [ ]:
import pandas as pd
import os
model = 'moment'

df = pd.read_csv('../data/IMU_10Hz_task_labeled.csv')
df_label = df

df.rename(columns={
    "accX": "accel_x","accY": "accel_y","accZ": "accel_z",
    "gyroX": "gyro_x","gyroY": "gyro_y","gyroZ": "gyro_z",
    "magX": "mag_x","magY": "mag_y","magZ": "mag_z"
    }, inplace=True)

# casper_collection
usecols=[
    'accel_x', 'accel_y', 'accel_z',
    'gyro_x', 'gyro_y', 'gyro_z',
    'mag_x', 'mag_y', 'mag_z',
    ]

df = df[usecols]
df = df[:100000]

abnormal = 'casper2'
result_df = run_moment(df=df, abnormal=abnormal)

cd_label = df_label[512:]['cd_label']
result_df['cd_label'] = cd_label

# Save result
save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_normal_benchmark_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

###### **hitting**

In [ ]:
import pandas as pd
import os
model = 'moment'

df = pd.read_csv('../data/IMU_hitting_arm_task_labeled.csv')

df.rename(columns={
    "accX": "accel_x","accY": "accel_y","accZ": "accel_z",
    "gyroX": "gyro_x","gyroY": "gyro_y","gyroZ": "gyro_z",
    "magX": "mag_x","magY": "mag_y","magZ": "mag_z"
    }, inplace=True)

# casper_collection
usecols=[
    'accel_x', 'accel_y', 'accel_z',
    'gyro_x', 'gyro_y', 'gyro_z',
    'mag_x', 'mag_y', 'mag_z',
    ]

df = df[usecols]

abnormal = 'casper2'
result_df = run_moment(df=df, abnormal=abnormal)
# Save result
save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_benchmark_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")

##### RoAD

In [ ]:
import pandas as pd
import os

usecols = [
    'sensor_id5_accel_x','sensor_id5_accel_y','sensor_id5_accel_z',
    'sensor_id5_gyro_x','sensor_id5_gyro_y','sensor_id5_gyro_z',
    'sensor_id6_accel_x','sensor_id6_accel_y','sensor_id6_accel_z',
    'sensor_id6_gyro_x','sensor_id6_gyro_y','sensor_id6_gyro_z',
    'label'
    ]
model = 'moment'
df = pd.read_csv("../data/road_collision.csv", usecols=usecols)

abnormal = 'road_collision'
result_df = run_moment(df=df, abnormal=abnormal)
# Save result
save_dir = f"../results/inference_df/zeroshot/{abnormal}_all_scenario/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{model}_benchmark_output.csv")
result_df.to_csv(save_path, index=False)
print(f"Saved to {save_path}")